# 03 Feature Engineering

## Objective

This notebook creates the Gold analytical layer for the Olist Customer Analytics project.

The Gold layer contains business-ready tables used for customer analytics, RFM segmentation, and dashboarding.

Main steps:

- Load Silver datasets.
- Create analytical dimensions.
- Create order-level and order-item-level fact tables.
- Build customer-level features.
- Calculate RFM metrics.
- Assign customer segments.
- Save Gold tables as Parquet files.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

## 2. Define paths

This notebook assumes it is located inside:

`01_customer_analytics/notebooks/`

Therefore, `..` points to the `01_customer_analytics/` project folder.

In [ ]:
PROJECT_PATH = Path("..")
SILVER_PATH = PROJECT_PATH / "data" / "silver"
GOLD_PATH = PROJECT_PATH / "data" / "gold"

GOLD_PATH.mkdir(parents=True, exist_ok=True)

SILVER_PATH, GOLD_PATH

## 3. Load Silver datasets

In [ ]:
customers = pd.read_parquet(SILVER_PATH / "customers.parquet")
orders = pd.read_parquet(SILVER_PATH / "orders.parquet")
order_items = pd.read_parquet(SILVER_PATH / "order_items.parquet")
payments = pd.read_parquet(SILVER_PATH / "payments.parquet")
reviews = pd.read_parquet(SILVER_PATH / "reviews.parquet")
products = pd.read_parquet(SILVER_PATH / "products.parquet")
sellers = pd.read_parquet(SILVER_PATH / "sellers.parquet")

## 4. Inspect loaded tables

In [ ]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
}

summary = []

for table_name, df in tables.items():
    summary.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

pd.DataFrame(summary)

## 5. Create helper aggregations

These intermediate tables summarize order items, payments, reviews, and product categories at the order level.

In [ ]:
order_items_agg = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        items_count=("order_item_id", "count"),
        sellers_count=("seller_id", "nunique"),
        products_count=("product_id", "nunique")
    )
)

order_items_agg.head()

In [ ]:
payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_methods_count=("payment_type", "nunique")
    )
)

main_payment_type = (
    payments
    .sort_values(["order_id", "payment_value"], ascending=[True, False])
    .drop_duplicates("order_id")[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_agg = payments_agg.merge(main_payment_type, on="order_id", how="left")

payments_agg.head()

In [ ]:
reviews_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "nunique")
    )
)

reviews_agg.head()

In [ ]:
order_categories = (
    order_items
    .merge(
        products[["product_id", "product_category_name_english"]],
        on="product_id",
        how="left"
    )
)

order_categories_agg = (
    order_categories
    .groupby("order_id", as_index=False)
    .agg(
        categories_count=("product_category_name_english", "nunique")
    )
)

main_order_category = (
    order_categories
    .groupby(["order_id", "product_category_name_english"], as_index=False)
    .agg(category_items_count=("order_item_id", "count"))
    .sort_values(["order_id", "category_items_count"], ascending=[True, False])
    .drop_duplicates("order_id")[["order_id", "product_category_name_english"]]
    .rename(columns={"product_category_name_english": "main_category"})
)

order_categories_agg = order_categories_agg.merge(
    main_order_category,
    on="order_id",
    how="left"
)

order_categories_agg.head()

## 6. Create `fact_orders`

In [ ]:
fact_orders = (
    orders
    .merge(
        customers[["customer_id", "customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"]],
        on="customer_id",
        how="left"
    )
    .merge(order_items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(order_categories_agg, on="order_id", how="left")
)

fact_orders["total_order_value"] = fact_orders["order_value"] + fact_orders["freight_value"]

fact_orders.head()

## 7. Create `fact_order_items`

In [ ]:
fact_order_items = (
    order_items
    .merge(
        products[["product_id", "product_category_name", "product_category_name_english"]],
        on="product_id",
        how="left"
    )
    .merge(
        sellers[["seller_id", "seller_city", "seller_state"]],
        on="seller_id",
        how="left"
    )
)

fact_order_items.head()

## 8. Create dimensions

In [ ]:
customer_order_dates = (
    fact_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
        total_orders=("order_id", "nunique")
    )
)

dim_customers = (
    customers
    .sort_values("customer_id")
    .drop_duplicates("customer_unique_id")
    [["customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"]]
    .merge(customer_order_dates, on="customer_unique_id", how="left")
)

dim_customers.head()

In [ ]:
dim_products = products.copy()
dim_sellers = sellers.copy()

dim_products.head()

## 9. Create customer-level feature table

This table consolidates behavior, value, satisfaction, delivery experience, and category-level metrics at the customer level.

In [ ]:
customer_basic_features = (
    fact_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_city=("customer_city", "first"),
        customer_state=("customer_state", "first"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
        total_orders=("order_id", "nunique"),
        total_spent=("total_order_value", "sum"),
        total_product_value=("order_value", "sum"),
        total_freight_value=("freight_value", "sum"),
        total_items=("items_count", "sum"),
        avg_review_score=("review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        delayed_orders=("is_delayed", "sum"),
        categories_bought=("categories_count", "sum")
    )
)

customer_basic_features["avg_order_value"] = (
    customer_basic_features["total_spent"] / customer_basic_features["total_orders"]
)

customer_basic_features["delay_rate"] = (
    customer_basic_features["delayed_orders"] / customer_basic_features["total_orders"]
)

customer_basic_features.head()

In [ ]:
preferred_payment_type = (
    fact_orders
    .groupby(["customer_unique_id", "main_payment_type"], as_index=False)
    .agg(payment_type_orders=("order_id", "nunique"))
    .sort_values(["customer_unique_id", "payment_type_orders"], ascending=[True, False])
    .drop_duplicates("customer_unique_id")
    [["customer_unique_id", "main_payment_type"]]
    .rename(columns={"main_payment_type": "preferred_payment_type"})
)

favorite_category = (
    fact_orders
    .groupby(["customer_unique_id", "main_category"], as_index=False)
    .agg(category_orders=("order_id", "nunique"))
    .sort_values(["customer_unique_id", "category_orders"], ascending=[True, False])
    .drop_duplicates("customer_unique_id")
    [["customer_unique_id", "main_category"]]
    .rename(columns={"main_category": "favorite_category"})
)

customer_features = (
    customer_basic_features
    .merge(preferred_payment_type, on="customer_unique_id", how="left")
    .merge(favorite_category, on="customer_unique_id", how="left")
)

customer_features.head()

## 10. Create RFM table

RFM stands for Recency, Frequency, and Monetary value.

Because this is a historical dataset, the snapshot date is defined as one day after the latest purchase date in the dataset.

In [ ]:
snapshot_date = fact_orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

customer_rfm = (
    fact_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        last_order_date=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary_value=("total_order_value", "sum")
    )
)

customer_rfm["recency_days"] = (
    snapshot_date - customer_rfm["last_order_date"]
).dt.days

customer_rfm.head()

## 11. Calculate RFM scores

Higher scores represent better customer behavior:

- Higher Recency score = more recent purchase.
- Higher Frequency score = more orders.
- Higher Monetary score = higher total spending.

In [ ]:
customer_rfm["r_score"] = pd.qcut(
    customer_rfm["recency_days"],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

customer_rfm["f_score"] = pd.qcut(
    customer_rfm["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

customer_rfm["m_score"] = pd.qcut(
    customer_rfm["monetary_value"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

customer_rfm["rfm_score"] = (
    customer_rfm["r_score"].astype(str)
    + customer_rfm["f_score"].astype(str)
    + customer_rfm["m_score"].astype(str)
)

customer_rfm.head()

## 12. Assign customer segments

The Olist dataset has a large share of one-time buyers. Because of that, the segmentation includes a specific segment for high-value one-time buyers.

In [ ]:
def assign_customer_segment(row):
    r = int(row["r_score"])
    f = int(row["f_score"])
    m = int(row["m_score"])
    frequency = int(row["frequency"])

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif frequency == 1 and m >= 4:
        return "High Value One-time Buyers"
    elif r >= 4 and f <= 2:
        return "Recent Customers"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Lost Customers"
    elif frequency == 1:
        return "One-time Buyers"
    else:
        return "Regular Customers"


customer_segments = customer_rfm.copy()
customer_segments["customer_segment"] = customer_segments.apply(assign_customer_segment, axis=1)

customer_segments.head()

## 13. Inspect segment distribution

In [ ]:
segment_distribution = (
    customer_segments
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary_value=("monetary_value", "mean"),
        total_monetary_value=("monetary_value", "sum")
    )
    .sort_values("customers", ascending=False)
)

segment_distribution

## 14. Save Gold tables

In [ ]:
fact_orders.to_parquet(GOLD_PATH / "fact_orders.parquet", index=False)
fact_order_items.to_parquet(GOLD_PATH / "fact_order_items.parquet", index=False)
dim_customers.to_parquet(GOLD_PATH / "dim_customers.parquet", index=False)
dim_products.to_parquet(GOLD_PATH / "dim_products.parquet", index=False)
dim_sellers.to_parquet(GOLD_PATH / "dim_sellers.parquet", index=False)
customer_features.to_parquet(GOLD_PATH / "customer_features.parquet", index=False)
customer_rfm.to_parquet(GOLD_PATH / "customer_rfm.parquet", index=False)
customer_segments.to_parquet(GOLD_PATH / "customer_segments.parquet", index=False)

## 15. Validate Gold files

In [ ]:
gold_files = sorted(GOLD_PATH.glob("*.parquet"))

for file in gold_files:
    print(file.name)

## Conclusion

The Gold analytical layer was created successfully.

The project now has customer-level features, RFM metrics, and customer segments ready for analysis and dashboarding.